# DeepSeek-OCR + QLoRA for **Sinhala Handwritten** OCR

Fine-tunes **`unsloth/DeepSeek-OCR`** (DeepSeek-OCR V1) with **QLoRA** on
`Datasets/handwritten-data/train` (908 crops) and evaluates on
`Datasets/handwritten-data/test` (227 crops).

**Goal: beat CER 0.5253**, the best handwritten Sinhala result from the author's own prior work
(`GuideForTrOCRLiLTXlmr/Resources/SinFUND_SinOCR_IJDAR.pdf`).

---

## Why this is a real research contribution

`GuideForDeepSeek/Resources/deepSeekOCR.pdf` (= arXiv:2606.29378, *Cross-Temporal Sinhala OCR*)
fine-tuned DeepSeek-OCR for Sinhala, but **only on printed, page-level documents** — 1,010 scanned
pages of Sri Lankan Legislative Acts. Its own Limitations section says:

> *"The corpus contains only Sri Lankan legislative statutes... The performance could vary widely
> across Sinhala in newspapers, textbooks, and **handwritten Sinhala**."*

So this notebook attacks a gap that paper explicitly names as untested, along **two** axes:

| Axis | arXiv:2606.29378 | This work |
|---|---|---|
| Script type | printed / scanned | **handwritten** |
| Granularity | full page | **word / line crops** |

## The evaluation is exactly comparable (verified, not assumed)

`Datasets/handwritten-data` **is** the *SinOCR-Handwritten* dataset from the author's IJDAR paper.
Table 2 of that paper reports train 908 / test 227 images with 17,122 / 4,029 total characters;
the CSVs here measure **908 / 227 images and 17,122 / 4,029 Unicode code points — exact matches**.
Cell 2 re-asserts this at runtime.

Consequences:
1. Same dataset, same official split, same 227 test images ⇒ the 0.5253 comparison is apples-to-apples.
2. Because the paper's character totals equal the **code-point** count exactly, its
   `CER = (S+D+I)/N` (Equation 1, `N` = reference characters) is measured on code points.
   This notebook uses that identical definition as its primary metric.
3. ~34% of test rows share their text string with some train row. That is a property of the
   *published* split, and TrOCR's 0.5253 was measured under identical conditions — so it does not
   threaten the comparison. Cell 3 tags every test row so results are also reported split by
   **seen-text vs unseen-text**.

### Baselines on this exact 227-image test set (IJDAR Table 5)

| Model | Handwritten CER |
|---|---|
| TrOCR (printed only) | 0.9940 |
| Tesseract (pre-trained) | 0.9493 |
| Google Vision API | 0.7532 |
| Tesseract (printed + handwritten) | 0.7204 |
| **TrOCR (printed → handwritten) — the bar to beat** | **0.5253** |

---

## Design decisions, and the evidence for each

| Decision | Choice | Evidence |
|---|---|---|
| Base model | **DeepSeek-OCR V1** (`unsloth/DeepSeek-OCR`) | arXiv:2606.29378 Tables III/IV: V1 zero-shot CER 0.6146 vs V2's 0.9611, fine-tuned 0.0302 vs 0.0694. V1 has a far stronger Sinhala prior. |
| **Warm start** | Continue training **`avishadilhara/sinhala-deepseek-ocr-Qlora`** | The author's own paper proves two-stage training is what makes this data work: TrOCR printed-only scored 0.9940 on handwriting; printed→handwritten scored **0.5253**. Print pre-training was worth ~0.47 CER. We inherit a *better* print model (CER 0.0302 vs TrOCR's 0.1445 on print). |
| Resolution | **Gundam: `base_size=1024, image_size=640, crop_mode=True`** | arXiv:2606.29378 §IV-A: *"All experiments used Gundam mode (base size 1024 px, crop size 640 px)"*. Also matches what the warm-start adapter was trained at. Measured on this dataset: 64% of images need no tiling (273 image tokens); the rest tile to ≤9×1 (1183 tokens); the text strip lands at a median 178 px tall inside the 1024 canvas — enough to resolve diacritics. |
| Adapt the vision tower? | **Yes** (fresh LoRA on SAM + CLIP + projector) | print→handwriting is a large *visual* domain shift, and the warm-start adapter only touches decoder projections. TrOCR needed full encoder training to reach 0.5253. |
| Quantisation | **4-bit NF4 (QLoRA)** | Matches the technique in the research question and the paper's Experiment 1 (P100 16 GB, 4-bit NF4, r=16, α=16). |
| Compute dtype | **fp16, chosen by compute capability** | `modeling_deepseekocr.py:30` sets its dtype from `torch.cuda.is_bf16_supported()`, which returns **True on a T4/P100 via emulation** even though neither has native bf16. Cell 1 selects by `get_device_capability()[0] >= 8` and overrides that module global. |
| Attention impl | default (SDPA) — **no flash-attn** | `deepencoder.py:522` sets `use_flash_attn=False`, and *both* branches of `NoTPAttention.forward` call `scaled_dot_product_attention` anyway. Nothing requires flash-attn, which is unavailable on Turing/Pascal. |
| Inference path | **custom, sharing the collator's preprocessing** | `model.infer()` returns text only with `eval_mode=True`, hardcodes `max_new_tokens=8192`, and applies `format_messages` — which the training collator does **not**. That makes the training prompt (`"…Free OCR. "`) differ from the inference prompt (`"…Free OCR."`) by a trailing space. Our eval builds the sequence with the *same code path as training*, so train/eval parity is exact and asserted in Cell 8. |

## Running this on Kaggle

- **Accelerator: `GPU T4 ×2`** (the notebook pins GPU 0; a single T4 is enough).
  **Do NOT pick `GPU P100`** — a real run proved it fails: P100 is `sm_60`, and Kaggle's
  torch 2.10.0+cu128 only ships kernels for `sm_70`–`sm_120`, so every CUDA op dies with
  *"no kernel image is available for execution on the device"*. 4-bit bitsandbytes needs
  `sm_75+` anyway. **Do not pick TPU** — bitsandbytes/PEFT need CUDA. Cell 1 now fails
  immediately on an unsupported GPU instead of dying mid-download.
- **Internet: ON** (downloads the base model and the warm-start adapter).
- **Input data:** attach `handwritten-data2`. Discovery is by directory *structure*, so the mount path can change.
- No `HF_TOKEN` needed — every repo used here is public.
- Runtime: roughly 4–7 h for the default 8 epochs plus baselines and final eval. A wall-clock guard
  stops training early so the evaluation cells always run.


In [ ]:
# ============================================================================
# Cell 1 — Configuration, environment, and the fp16/bf16 decision
# ============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # single GPU on purpose
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Pin the stack. transformers 4.56.2 is what Unsloth's official DeepSeek-OCR
# recipe pins; the custom modeling code predates transformers v5 and breaks on it.
!pip install -q -U "transformers==4.56.2" "peft>=0.13.0" "accelerate>=0.34.0" bitsandbytes
!pip install -q einops addict easydict jiwer
# torchao: PEFT's LoRA dispatcher calls is_torchao_available(), which RAISES (rather than
# returning False) when torchao is present but older than 0.16.0. Kaggle ships 0.10.0, which
# killed get_peft_model() in Cell 5 on a real run. --no-deps so it cannot drag in a different
# torch and break the CUDA build. We never use torchao (quantisation is bitsandbytes NF4).
!pip install -q --no-deps --upgrade "torchao>=0.16.0"

import sys, random, json, math, time, gc, csv, copy
import numpy as np
import torch

# ---------------------------------------------------------------- CONFIG ----
CFG = dict(
    # --- what to run (flip these to slice the notebook) -------------------
    smoke_test            = False,  # True => tiny end-to-end run to validate the pipeline
    run_zero_shot         = True,   # measure the un-finetuned baselines first
    do_train              = True,
    run_final_test_eval   = True,

    # --- model ------------------------------------------------------------
    base_repo             = "unsloth/DeepSeek-OCR",
    print_adapter_repo    = "avishadilhara/sinhala-deepseek-ocr-Qlora",
    warm_start            = True,   # continue the Sinhala PRINT adapter (two-stage recipe)
    adapt_vision_tower    = True,   # add fresh LoRA to SAM + CLIP + projector
    load_in_4bit          = True,   # QLoRA
    lora_r                = 16,     # MUST be 16 while warm_start=True (adapter shapes)
    lora_alpha            = 16,
    lora_dropout          = 0.05,   # >0 for regularisation on a small train set

    # --- input representation (Gundam; see header table) ------------------
    base_size             = 1024,
    image_size            = 640,
    crop_mode             = True,
    prompt                = "<image>\nFree OCR. ",

    # --- data -------------------------------------------------------------
    val_fraction          = 0.10,   # carved out of TRAIN; test is never touched until the end
    augment               = True,

    # --- optimisation -----------------------------------------------------
    epochs                = 8,
    per_device_batch      = 1,
    grad_accum            = 8,      # effective batch 8
    lr                    = 1e-4,
    warmup_ratio          = 0.03,
    weight_decay          = 0.01,
    max_grad_norm         = 0.3,
    grad_checkpointing    = True,

    # --- validation / early stopping --------------------------------------
    eval_every_steps      = 50,
    val_eval_n            = 64,     # generation is slow; score this many val samples per eval
    early_stop_patience   = 4,      # evals without val-CER improvement before stopping

    # --- budget guards ----------------------------------------------------
    train_hours_budget    = 6.5,    # stop training after this, so eval cells always run
    eval_max_new_tokens   = 0,      # 0 => derive from the measured target-length distribution
    zeroshot_n            = 227,    # test samples for each zero-shot row (227 = all)
    zeroshot_max_new_tok  = 192,    # un-finetuned models ramble; bound the runtime
    seed                  = 42,

    # --- integrity --------------------------------------------------------
    assert_dataset_identity = True, # hard-check against IJDAR Table 2 (908/227, 17122/4029)
)
if CFG["smoke_test"]:
    CFG.update(epochs=1, zeroshot_n=6, val_eval_n=6, eval_every_steps=5,
               train_hours_budget=0.5)
    print(">>> SMOKE TEST MODE: tiny run, results are NOT meaningful\n")

IN_KAGGLE  = os.path.isdir("/kaggle")
OUTPUT_DIR = "/kaggle/working" if IN_KAGGLE else "."
RUN_DIR    = os.path.join(OUTPUT_DIR, "deepseek_hw_run")
BEST_DIR   = os.path.join(RUN_DIR, "best_adapter")
os.makedirs(RUN_DIR, exist_ok=True)

# ------------------------------------------------------- GPU: fail fast ----
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU. This trains a ~3B-parameter VLM and is not viable on CPU.\n"
        "On Kaggle: Settings -> Accelerator -> 'GPU T4 x2' (or 'GPU P100'). NOT TPU."
    )

cap = torch.cuda.get_device_capability(0)
HAS_NATIVE_BF16 = cap[0] >= 8
COMPUTE_DTYPE   = torch.bfloat16 if HAS_NATIVE_BF16 else torch.float16
USE_FP16_AMP    = not HAS_NATIVE_BF16

print(f"GPU              : {torch.cuda.get_device_name(0)}  (compute capability {cap[0]}.{cap[1]})")
print(f"torch / CUDA     : {torch.__version__} / {torch.version.cuda}")
print(f"total VRAM       : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"is_bf16_supported: {torch.cuda.is_bf16_supported()}   <-- can be True via EMULATION")
print(f"native bf16      : {HAS_NATIVE_BF16}")
print(f"=> COMPUTE_DTYPE : {COMPUTE_DTYPE}   (AMP fp16={USE_FP16_AMP})")
if not HAS_NATIVE_BF16:
    print("   Turing/Pascal detected: forcing fp16. modeling_deepseekocr.py:30 would otherwise")
    print("   pick bf16 from is_bf16_supported() and run emulated/slow. Overridden in Cell 4.")

# ---- the installed PyTorch must actually HAVE KERNELS for this GPU ---------
# A real Kaggle run on a Tesla P100 (sm_60) printed only a UserWarning here and then
# died four cells later, mid-model-load, with "CUDA error: no kernel image is available
# for execution on the device" -- after downloading 6.7 GB of weights. Kaggle's
# torch 2.10.0+cu128 ships kernels for sm_70..sm_120 only. Fail here instead.
ARCH_LIST = torch.cuda.get_arch_list()
DEV_ARCH  = f"sm_{cap[0]}{cap[1]}"
print(f"\ntorch built for arch: {ARCH_LIST}")
print(f"this GPU requires   : {DEV_ARCH}")
if DEV_ARCH not in ARCH_LIST:
    raise RuntimeError(
        f"This PyTorch build ({torch.__version__}) has NO compiled kernels for {DEV_ARCH} "
        f"({torch.cuda.get_device_name(0)}).\n"
        f"  supported: {ARCH_LIST}\n\n"
        f"FIX: Kaggle -> Settings -> Accelerator -> 'GPU T4 x2'   (T4 is sm_75).\n"
        f"     Do NOT use 'GPU P100' (sm_60): unusable with this PyTorch, and 4-bit\n"
        f"     bitsandbytes requires sm_75+ regardless. Do NOT use TPU."
    )

# empirically prove a kernel really launches, rather than trusting the arch list
try:
    _probe = torch.randn(64, 64, device="cuda", dtype=COMPUTE_DTYPE)
    _ = (_probe @ _probe).sum().item()
    del _probe
    torch.cuda.synchronize()
    print(f"CUDA smoke test     : OK ({COMPUTE_DTYPE} matmul executed on device)")
except Exception as e:
    raise RuntimeError(
        f"A trivial {COMPUTE_DTYPE} matmul failed on {torch.cuda.get_device_name(0)}: {e}\n"
        f"Switch the Kaggle accelerator to 'GPU T4 x2'."
    )

# ------------------------------------------------------------ determinism --
# ---- torchao must be new enough for PEFT, or absent entirely --------------
import importlib, importlib.metadata, subprocess
from packaging.version import parse as _ver
def _torchao_version():
    importlib.invalidate_caches()
    try:
        return _ver(importlib.metadata.version("torchao"))
    except importlib.metadata.PackageNotFoundError:
        return None
_ta = _torchao_version()
if _ta is not None and _ta < _ver("0.16.0"):
    print(f"torchao {_ta} < 0.16.0 and the upgrade did not take -> removing it "
          f"(PEFT then skips its dispatcher cleanly).")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    _ta = _torchao_version()
if _ta is not None and _ta < _ver("0.16.0"):
    raise RuntimeError(
        f"torchao {_ta} is installed and cannot be upgraded or removed. PEFT's "
        f"is_torchao_available() will raise ImportError inside get_peft_model() (Cell 5). "
        f"Restart the Kaggle session and re-run this cell."
    )
print(f"torchao: {_ta if _ta else 'not installed'} -> PEFT will "
      f"{'use' if _ta else 'skip'} its torchao dispatcher")

SEED = CFG["seed"]
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print(f"\nrun dir: {RUN_DIR}")
print(json.dumps({k: (str(v) if not isinstance(v, (int, float, bool, str, type(None))) else v)
                  for k, v in CFG.items()}, indent=2))


In [ ]:
# ============================================================================
# Cell 2 — Find the dataset by STRUCTURE, load it, verify it is SinOCR-Handwritten
# ============================================================================
import os, csv

REQUIRED = [("train", "data.csv"), ("train", "images"), ("test", "data.csv"), ("test", "images")]
KNOWN_HINT = "/kaggle/input/datasets/danushamsc25/handwritten-data2"   # verified, not trusted

def find_dataset_root(roots, max_depth=6):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, _ in os.walk(root):
            if dirpath[len(root):].count(os.sep) > max_depth:
                dirnames[:] = []
                continue
            if all(os.path.exists(os.path.join(dirpath, a, b)) for a, b in REQUIRED):
                found.append(dirpath)
                dirnames[:] = []
    # de-duplicate while preserving order
    return list(dict.fromkeys(found))

def print_tree(root, max_depth=4):
    if not os.path.isdir(root):
        print(f"(no such directory: {root})"); return
    for dirpath, dirnames, filenames in os.walk(root):
        d = dirpath[len(root):].count(os.sep)
        if d > max_depth:
            dirnames[:] = []; continue
        print("  " * d + (os.path.basename(dirpath) or dirpath) + "/")
        if d == max_depth:
            for fn in sorted(filenames)[:15]:
                print("  " * (d + 1) + fn)

cands = find_dataset_root([KNOWN_HINT, "/kaggle/input", ".", "/kaggle/working"])
if not cands:
    print("Could not find the dataset (need train/ and test/, each with data.csv + images/).")
    print("\nTree of /kaggle/input:")
    print_tree("/kaggle/input")
    raise FileNotFoundError("Attach the `handwritten-data2` dataset (Add Input -> Datasets), then re-run.")

DATASET_ROOT = cands[0]
print(f"dataset root: {DATASET_ROOT}" + (f"   (other matches: {cands[1:]})" if len(cands) > 1 else ""))

def load_split(root, split):
    csv_path   = os.path.join(root, split, "data.csv")
    images_dir = os.path.join(root, split, "images")
    samples, missing = [], []
    with open(csv_path, encoding="utf-8") as f:
        for r in csv.DictReader(f):
            p = os.path.join(images_dir, f"{r['file_name']}.png")
            (samples if os.path.exists(p) else missing).append(
                {"id": r["file_name"], "image": p, "text": r["text"]} if os.path.exists(p) else p)
    return samples, missing

TRAIN_ALL, miss_tr = load_split(DATASET_ROOT, "train")
TEST,      miss_te = load_split(DATASET_ROOT, "test")
if miss_tr or miss_te:
    raise FileNotFoundError(f"Missing images: {len(miss_tr)} train, {len(miss_te)} test. e.g. {(miss_tr + miss_te)[:3]}")

# ---- integrity check against IJDAR Table 2 (this is what makes 0.5253 comparable) ----
IJDAR_TABLE2 = {"train": {"n": 908, "chars": 17122}, "test": {"n": 227, "chars": 4029}}
print("\nIntegrity check vs IJDAR Table 2 (SinOCR-Handwritten):")
ok_all = True
for name, rows in (("train", TRAIN_ALL), ("test", TEST)):
    n     = len(rows)
    chars = sum(len(r["text"]) for r in rows)          # Unicode code points, as the paper counts
    exp   = IJDAR_TABLE2[name]
    ok    = (n == exp["n"] and chars == exp["chars"])
    ok_all &= ok
    print(f"  {name:5}: images {n:5} (paper {exp['n']:5})   code points {chars:6} (paper {exp['chars']:6})   "
          f"{'MATCH' if ok else 'MISMATCH'}")
if ok_all:
    print("  => This IS the published SinOCR-Handwritten split. CER is directly comparable to 0.5253.")
elif CFG["assert_dataset_identity"]:
    raise AssertionError(
        "Dataset does not match IJDAR Table 2. The 0.5253 comparison would NOT be apples-to-apples.\n"
        "Set CFG['assert_dataset_identity']=False only if you deliberately changed the dataset."
    )

# ---- measured statistics (never assumed) ----
from PIL import Image
def stats(rows, label):
    L = sorted(len(r["text"]) for r in rows)
    W = sorted(r["text"].split().__len__() for r in rows)
    q = lambda v, p: v[min(len(v) - 1, int(p * len(v)))]
    print(f"\n{label} (n={len(rows)})")
    print(f"  chars/sample : min {L[0]}  med {q(L,.5)}  p95 {q(L,.95)}  max {L[-1]}")
    print(f"  words/sample : min {W[0]}  med {q(W,.5)}  max {W[-1]}")
    sizes = [Image.open(r["image"]).size for r in rows]
    ws = sorted(s[0] for s in sizes); hs = sorted(s[1] for s in sizes)
    ars = sorted(s[0] / s[1] for s in sizes)
    print(f"  width  px    : min {ws[0]}  med {q(ws,.5)}  p95 {q(ws,.95)}  max {ws[-1]}")
    print(f"  height px    : min {hs[0]}  med {q(hs,.5)}  p95 {q(hs,.95)}  max {hs[-1]}")
    print(f"  aspect ratio : min {ars[0]:.1f}  med {q(ars,.5):.1f}  p95 {q(ars,.95):.1f}  max {ars[-1]:.1f}")
    tile = sum(1 for s in sizes if s[0] > 640 or s[1] > 640)
    print(f"  will TILE (any dim > 640): {tile}/{len(rows)} ({100*tile/len(rows):.1f}%) "
          f"-> those cost 1183 image tokens, the rest 273")
    return sizes

stats(TRAIN_ALL, "TRAIN (all)")
stats(TEST, "TEST")
print("\nexample:", {k: v for k, v in TRAIN_ALL[0].items()})


In [ ]:
# ============================================================================
# Cell 3 — Train/val split (val comes out of TRAIN) + seen/unseen-text tagging
# ============================================================================
import random

# The TEST split is used exactly once, in the final cell. Validation for checkpoint
# selection and early stopping is carved out of TRAIN so nothing selects on test.
rng = random.Random(CFG["seed"])

# stratify by text-length bucket so val mirrors train's length distribution
def bucket(t):
    n = len(t)
    return 0 if n < 10 else 1 if n < 20 else 2 if n < 35 else 3

by_bucket = {}
for r in TRAIN_ALL:
    by_bucket.setdefault(bucket(r["text"]), []).append(r)

TRAIN, VAL = [], []
for b, rows in sorted(by_bucket.items()):
    rows = rows[:]                       # copy before shuffling
    rng.shuffle(rows)
    k = max(1, int(round(CFG["val_fraction"] * len(rows))))
    VAL.extend(rows[:k]); TRAIN.extend(rows[k:])
rng.shuffle(TRAIN); rng.shuffle(VAL)

assert len({r["id"] for r in TRAIN} & {r["id"] for r in VAL}) == 0, "train/val id overlap"
test_ids = {r["id"] for r in TEST}
assert len({r["id"] for r in TRAIN + VAL} & test_ids) == 0, "TRAIN/VAL LEAKED INTO TEST"
print(f"train {len(TRAIN)}   val {len(VAL)}   test {len(TEST)}  (val carved from train; test untouched)")
print("assert passed: no id from train/val appears in test")

# --- tag test rows as seen-text / unseen-text, for split reporting ---------
train_texts_used = {r["text"].strip() for r in TRAIN}          # what the model actually trains on
train_texts_all  = {r["text"].strip() for r in TRAIN_ALL}      # what TrOCR trained on (all 908)
for r in TEST:
    r["seen_in_train_used"] = r["text"].strip() in train_texts_used
    r["seen_in_train_all"]  = r["text"].strip() in train_texts_all

n_seen_used = sum(r["seen_in_train_used"] for r in TEST)
n_seen_all  = sum(r["seen_in_train_all"]  for r in TEST)
print(f"\ntest rows whose exact text also occurs in the data we train on : {n_seen_used}/{len(TEST)} "
      f"({100*n_seen_used/len(TEST):.1f}%)")
print(f"test rows whose exact text occurs anywhere in the 908 train rows: {n_seen_all}/{len(TEST)} "
      f"({100*n_seen_all/len(TEST):.1f}%)  <- same condition TrOCR's 0.5253 was measured under")
print("Final metrics are reported overall AND split by seen/unseen text, so this is disclosed rather than hidden.")

if CFG["smoke_test"]:
    TRAIN, VAL = TRAIN[:16], VAL[:6]
    print(f"\nSMOKE TEST: truncated to train {len(TRAIN)}, val {len(VAL)}")


In [ ]:
# ============================================================================
# Cell 4 — Download base model, patch its dtype global, load in 4-bit (QLoRA)
# ============================================================================
import sys, torch
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

BASE_DIR = "deepseek_ocr"          # same local dir name the warm-start adapter was trained against
snapshot_download(CFG["base_repo"], local_dir=BASE_DIR)     # public repo, no token needed

tokenizer = AutoTokenizer.from_pretrained(BASE_DIR, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"tokenizer: bos={tokenizer.bos_token_id} eos={tokenizer.eos_token_id} pad={tokenizer.pad_token_id}")

quant_cfg = None
if CFG["load_in_4bit"]:
    quant_cfg = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = "nf4",
        bnb_4bit_compute_dtype    = COMPUTE_DTYPE,
        bnb_4bit_use_double_quant = True,
    )

model = AutoModel.from_pretrained(
    BASE_DIR,
    trust_remote_code   = True,
    use_safetensors     = True,
    quantization_config = quant_cfg,
    device_map          = {"": 0},
    dtype               = COMPUTE_DTYPE,   # `torch_dtype=` is deprecated in transformers 4.56
    attn_implementation = "eager",     # no flash-attn on Turing/Pascal; SDPA is used internally anyway
)
# Expect one benign warning here: "You are using a model of type deepseek_vl_v2 to
# instantiate a model of type DeepseekOCR". config.json's model_type and its auto_map
# class name differ by design for this custom architecture; loading is still correct.
print(f"\nloaded {CFG['base_repo']} | 4bit={CFG['load_in_4bit']} | "
      f"VRAM allocated {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# ---- grab the dynamically-imported custom module and FIX ITS DTYPE GLOBAL ----
# modeling_deepseekocr.py:30 is
#     torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
# and is_bf16_supported() reports True on T4/P100 through emulation. Override it.
_mods = [m for m in sys.modules if m.endswith("modeling_deepseekocr")]
if not _mods:
    raise RuntimeError("Could not locate the dynamic modeling module; cannot verify its dtype.")
DSMOD = sys.modules[_mods[0]]
print(f"\ncustom module: {DSMOD.__name__}")
print(f"  its torch_dtype was : {DSMOD.torch_dtype}")
DSMOD.torch_dtype = COMPUTE_DTYPE
print(f"  overridden to       : {DSMOD.torch_dtype}")

# preprocessing helpers, taken from the *same* module instance the model uses
BasicImageTransform = DSMOD.BasicImageTransform
dynamic_preprocess  = DSMOD.dynamic_preprocess
text_encode         = DSMOD.text_encode
print("  imported BasicImageTransform / dynamic_preprocess / text_encode from it")

# ---- what the LoRA targets will actually hit (names verified in deepencoder.py) ----
import collections
suffix_counts = collections.Counter()
for name, mod in model.named_modules():
    if isinstance(mod, torch.nn.Linear) or mod.__class__.__name__.startswith("Linear4bit"):
        suffix_counts[name.split(".")[-1]] += 1
print("\nLinear-layer name suffixes present in the model (target_modules must match these):")
for k, v in suffix_counts.most_common(24):
    print(f"  {k:24} x{v}")


In [ ]:
# ============================================================================
# Cell 5 — Build the LoRA: warm-start the print adapter + fresh vision-tower LoRA
# ============================================================================
import torch
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# Decoder targets: exactly what avishadilhara/sinhala-deepseek-ocr-Qlora adapted.
DECODER_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# Vision targets, verified by reading deepencoder.py:
#   SAM ViT-B  Attention.qkv / .proj, MLPBlock.lin1 / .lin2
#   CLIP-L     NoTPAttention.qkv_proj / .out_proj, NoTPFeedForward.fc1 / .fc2
# (PEFT matches on `key.endswith("." + target)`, so "proj" cannot collide with "q_proj",
#  and "up_proj" cannot collide with the projector's "high_up_proj"/"low_up_proj".)
VISION_TARGETS = ["qkv", "proj", "lin1", "lin2", "qkv_proj", "out_proj", "fc1", "fc2"]

targets = list(DECODER_TARGETS) + (VISION_TARGETS if CFG["adapt_vision_tower"] else [])

if CFG["warm_start"] and CFG["lora_r"] != 16:
    raise ValueError("warm_start=True requires lora_r=16 to match the print adapter's tensor shapes.")

lora_cfg = LoraConfig(
    r              = CFG["lora_r"],
    lora_alpha     = CFG["lora_alpha"],
    lora_dropout   = CFG["lora_dropout"],
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = targets,
)
model = get_peft_model(model, lora_cfg)

n_lora_mods = sum(1 for n, _ in model.named_modules() if n.endswith("lora_A"))
print(f"LoRA modules created: {n_lora_mods}   (targets: {targets})")
if n_lora_mods == 0:
    raise RuntimeError("No LoRA modules were created — target_modules matched nothing.")

# ------------------------------------------------- warm start (two-stage) ----
WARM_START_OK = False
if CFG["warm_start"]:
    path = hf_hub_download(CFG["print_adapter_repo"], "adapter_model.safetensors")
    sd   = load_file(path)
    n_in_file = sum(1 for k in sd if "lora_" in k)

    load_result = set_peft_model_state_dict(model, sd, adapter_name="default")
    unexpected  = [k for k in load_result.unexpected_keys if "lora_" in k]

    # Missing keys are expected here (they are the fresh vision LoRAs and the whole
    # frozen base), so the meaningful check is: did every tensor in the file land?
    print(f"\nwarm start from {CFG['print_adapter_repo']}")
    print(f"  LoRA tensors in adapter file : {n_in_file}")
    print(f"  tensors that could NOT be placed: {len(unexpected)}")
    if unexpected:
        print("  first few unplaced:", unexpected[:5])
        raise RuntimeError(
            "Warm start FAILED: the print adapter's tensors do not fit this model. "
            "Set CFG['warm_start']=False to train a fresh LoRA instead."
        )

    # lora_B is zero-initialised, so any non-zero lora_B proves real weights arrived.
    nz_B = sum(1 for n, p in model.named_parameters()
               if "lora_B" in n and torch.count_nonzero(p).item() > 0)
    tot_B = sum(1 for n, _ in model.named_parameters() if "lora_B" in n)
    print(f"  non-zero lora_B tensors after load: {nz_B}/{tot_B}")
    if nz_B == 0:
        raise RuntimeError("Warm start loaded nothing: every lora_B is still zero.")
    WARM_START_OK = True
    print("  => decoder LoRA now carries the Sinhala PRINT adaptation; "
          f"{tot_B - nz_B} fresh (vision) LoRAs start at identity.")
else:
    print("\nwarm_start disabled: training a fresh LoRA from scratch (ablation config).")

# ------------------------------------------- gradient checkpointing + grads --
if CFG["grad_checkpointing"]:
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        print("\ngradient checkpointing: ON (non-reentrant)")
    except Exception as e:
        print(f"\n[warn] gradient checkpointing unavailable ({e}); continuing without it (more VRAM).")
    try:
        model.enable_input_require_grads()
        print("enable_input_require_grads: OK (needed for checkpointing with a frozen embedding)")
    except Exception as e:
        print(f"[warn] enable_input_require_grads failed: {e}")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\ntrainable params: {trainable:,} / {total:,} ({100*trainable/total:.3f}%)")
print(f"NOTE: the decoder is a 64-expert MoE, so gate/up/down_proj expand to thousands of LoRA")
print(f"      modules. With ~{len(TRAIN)} training samples this is a high capacity/data ratio —")
print(f"      early stopping on validation CER (Cell 10) is what keeps it honest.")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


In [ ]:
# ============================================================================
# Cell 6 — Data collator (Unsloth's DeepSeekOCRDataCollator) + handwriting augmentation
# ============================================================================
# The collator below is Unsloth's official DeepSeek-OCR collator, kept as-is apart from
# two deliberate changes, both noted inline:
#   (1) dtype comes from COMPUTE_DTYPE instead of model.dtype — with a 4-bit base,
#       model.dtype is unreliable, and AMP handles the mixed-precision matmuls.
#   (2) an optional augmentation hook is applied to the raw crop before any resizing.
import io, math, torch
from dataclasses import dataclass
from typing import Dict, List, Any, Tuple
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
from torch.nn.utils.rnn import pad_sequence
import random as _random


# ------------------------------------------------------------ augmentation --
def _bg_level(img: Image.Image) -> int:
    """Median intensity of the border ring: the paper background for these scans."""
    g = img.convert("L")
    w, h = g.size
    px = list(g.crop((0, 0, w, 1)).getdata()) + list(g.crop((0, h - 1, w, h)).getdata()) \
       + list(g.crop((0, 0, 1, h)).getdata()) + list(g.crop((w - 1, 0, w, h)).getdata())
    px.sort()
    return int(px[len(px) // 2]) if px else 255


def augment_crop(img: Image.Image, rnd: _random.Random) -> Image.Image:
    """Conservative handwriting augmentation.

    Sinhala diacritics are small; aggressive blur/erosion destroys the very marks that
    distinguish characters. Every magnitude here is deliberately mild.
    """
    bg = _bg_level(img)
    fill = (bg, bg, bg) if img.mode == "RGB" else bg

    # small rotation (scans are near-upright already)
    if rnd.random() < 0.7:
        img = img.rotate(rnd.uniform(-1.2, 1.2), resample=Image.BILINEAR,
                         expand=True, fillcolor=fill)
    # mild shear, as a affine transform
    if rnd.random() < 0.4:
        s = rnd.uniform(-0.04, 0.04)
        w, h = img.size
        img = img.transform((w + int(abs(s) * h), h), Image.AFFINE,
                            (1, s, -s * h if s < 0 else 0, 0, 1, 0),
                            resample=Image.BILINEAR, fillcolor=fill)
    # slight isotropic rescale (writing size varies between writers)
    if rnd.random() < 0.5:
        f = rnd.uniform(0.93, 1.07)
        img = img.resize((max(8, int(img.width * f)), max(8, int(img.height * f))), Image.LANCZOS)
    # brightness / contrast jitter (ink darkness and paper age vary)
    if rnd.random() < 0.6:
        img = ImageEnhance.Brightness(img).enhance(rnd.uniform(0.90, 1.10))
    if rnd.random() < 0.6:
        img = ImageEnhance.Contrast(img).enhance(rnd.uniform(0.88, 1.14))
    # very light blur, emulating scan softness
    if rnd.random() < 0.25:
        img = img.filter(ImageFilter.GaussianBlur(rnd.uniform(0.3, 0.7)))
    # stroke thickness: MinFilter thickens dark ink, MaxFilter thins it
    r = rnd.random()
    if r < 0.12:
        img = img.filter(ImageFilter.MinFilter(3))
    elif r < 0.24:
        img = img.filter(ImageFilter.MaxFilter(3))
    return img


# --------------------------------------------------------------- collator ---
@dataclass
class DeepSeekOCRDataCollator:
    tokenizer: Any
    dtype: Any
    image_size: int = 640
    base_size: int = 1024
    crop_mode: bool = True
    image_token_id: int = 128815
    train_on_responses_only: bool = True

    def __init__(self, tokenizer, dtype, image_size=640, base_size=1024, crop_mode=True,
                 train_on_responses_only=True, augment=False, seed=0):
        self.tokenizer = tokenizer
        self.dtype = dtype                     # change (1): explicit, not model.dtype
        self.image_size = image_size
        self.base_size = base_size
        self.crop_mode = crop_mode
        self.image_token_id = 128815
        self.train_on_responses_only = train_on_responses_only
        self.augment = augment                 # change (2)
        self._rnd = _random.Random(seed)
        self.image_transform = BasicImageTransform(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5),
                                                   normalize=True)
        self.patch_size = 16
        self.downsample_ratio = 4
        self.bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 0

    def deserialize_image(self, image_data) -> Image.Image:
        if isinstance(image_data, Image.Image):
            return image_data.convert("RGB")
        if isinstance(image_data, str):
            return Image.open(image_data).convert("RGB")
        if isinstance(image_data, dict) and "bytes" in image_data:
            return Image.open(io.BytesIO(image_data["bytes"])).convert("RGB")
        raise ValueError(f"Unsupported image format: {type(image_data)}")

    def process_image(self, image: Image.Image):
        images_list, images_crop_list, images_spatial_crop = [], [], []
        if self.crop_mode:
            if image.size[0] <= 640 and image.size[1] <= 640:
                crop_ratio, images_crop_raw = (1, 1), []
            else:
                images_crop_raw, crop_ratio = dynamic_preprocess(
                    image, min_num=2, max_num=9, image_size=self.image_size, use_thumbnail=False)
            global_view = ImageOps.pad(image, (self.base_size, self.base_size),
                                       color=tuple(int(x * 255) for x in self.image_transform.mean))
            images_list.append(self.image_transform(global_view).to(self.dtype))
            w_num, h_num = crop_ratio
            images_spatial_crop.append([w_num, h_num])
            if w_num > 1 or h_num > 1:
                for c in images_crop_raw:
                    images_crop_list.append(self.image_transform(c).to(self.dtype))
            nq  = math.ceil((self.image_size // self.patch_size) / self.downsample_ratio)
            nqb = math.ceil((self.base_size  // self.patch_size) / self.downsample_ratio)
            tok = ([self.image_token_id] * nqb + [self.image_token_id]) * nqb
            tok += [self.image_token_id]
            if w_num > 1 or h_num > 1:
                tok += ([self.image_token_id] * (nq * w_num) + [self.image_token_id]) * (nq * h_num)
        else:
            crop_ratio = (1, 1)
            images_spatial_crop.append([1, 1])
            if self.base_size <= 640:
                view = image.resize((self.base_size, self.base_size), Image.LANCZOS)
            else:
                view = ImageOps.pad(image, (self.base_size, self.base_size),
                                    color=tuple(int(x * 255) for x in self.image_transform.mean))
            images_list.append(self.image_transform(view).to(self.dtype))
            nq  = math.ceil((self.base_size // self.patch_size) / self.downsample_ratio)
            tok = ([self.image_token_id] * nq + [self.image_token_id]) * nq
            tok += [self.image_token_id]
        return images_list, images_crop_list, images_spatial_crop, tok, crop_ratio

    def process_single_sample(self, messages: List[Dict], augment=None) -> Dict[str, Any]:
        do_aug = self.augment if augment is None else augment
        images = []
        for m in messages:
            for img in (m.get("images") or []):
                if img is not None:
                    im = self.deserialize_image(img)
                    if do_aug:
                        im = augment_crop(im, self._rnd)
                    images.append(im)
        if not images:
            raise ValueError("No images in sample.")

        tokenized_str, images_seq_mask = [], []
        images_list, images_crop_list, images_spatial_crop = [], [], []
        prompt_token_count, assistant_started, image_idx = -1, False, 0

        tokenized_str.append(self.bos_id); images_seq_mask.append(False)

        for message in messages:
            role, content = message["role"], message["content"]
            if role == "<|Assistant|>":
                if not assistant_started:
                    prompt_token_count = len(tokenized_str)
                    assistant_started = True
                content = f"{content.strip()} {self.tokenizer.eos_token}"
            for i, text_sep in enumerate(content.split("<image>")):
                t = text_encode(self.tokenizer, text_sep, bos=False, eos=False)
                tokenized_str.extend(t); images_seq_mask.extend([False] * len(t))
                if i < len(content.split("<image>")) - 1:
                    if image_idx >= len(images):
                        raise ValueError("'<image>' token without a matching image.")
                    il, cl, sc, tok, _ = self.process_image(images[image_idx])
                    images_list.extend(il); images_crop_list.extend(cl); images_spatial_crop.extend(sc)
                    tokenized_str.extend(tok); images_seq_mask.extend([True] * len(tok))
                    image_idx += 1
        if image_idx != len(images):
            raise ValueError(f"{len(images)} images but {image_idx} '<image>' tokens used.")
        if not assistant_started:
            prompt_token_count = len(tokenized_str)

        images_ori = torch.stack(images_list, dim=0)
        images_crop = (torch.stack(images_crop_list, dim=0) if images_crop_list
                       else torch.zeros((1, 3, self.base_size, self.base_size), dtype=self.dtype))
        return {
            "input_ids": torch.tensor(tokenized_str, dtype=torch.long),
            "images_seq_mask": torch.tensor(images_seq_mask, dtype=torch.bool),
            "images_ori": images_ori,
            "images_crop": images_crop,
            "images_spatial_crop": torch.tensor(images_spatial_crop, dtype=torch.long),
            "prompt_token_count": prompt_token_count,
        }

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch = []
        for f in features:
            try:
                batch.append(self.process_single_sample(f["messages"]))
            except Exception as e:
                print(f"[collator] dropped a sample: {e}")
        if not batch:
            raise ValueError("No valid samples in batch")

        input_ids = pad_sequence([b["input_ids"] for b in batch], batch_first=True,
                                 padding_value=self.tokenizer.pad_token_id)
        images_seq_mask = pad_sequence([b["images_seq_mask"] for b in batch], batch_first=True,
                                       padding_value=False)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100      # ignore padding
        labels[images_seq_mask] = -100                            # never predict image tokens
        if self.train_on_responses_only:
            for i, b in enumerate(batch):
                if b["prompt_token_count"] > 0:
                    labels[i, :b["prompt_token_count"]] = -100    # train on the answer only
        return {
            "input_ids": input_ids,
            "attention_mask": (input_ids != self.tokenizer.pad_token_id).long(),
            "labels": labels,
            "images": [(b["images_crop"], b["images_ori"]) for b in batch],
            "images_seq_mask": images_seq_mask,
            "images_spatial_crop": torch.cat([b["images_spatial_crop"] for b in batch], dim=0),
        }


def to_conversation(row):
    return {"messages": [
        {"role": "<|User|>",      "content": CFG["prompt"], "images": [row["image"]]},
        {"role": "<|Assistant|>", "content": row["text"]},
    ]}

train_collator = DeepSeekOCRDataCollator(tokenizer, COMPUTE_DTYPE, CFG["image_size"], CFG["base_size"],
                                         CFG["crop_mode"], True, augment=CFG["augment"], seed=CFG["seed"])
eval_collator  = DeepSeekOCRDataCollator(tokenizer, COMPUTE_DTYPE, CFG["image_size"], CFG["base_size"],
                                         CFG["crop_mode"], True, augment=False, seed=CFG["seed"])
print(f"collators ready | dtype={COMPUTE_DTYPE} | Gundam base={CFG['base_size']} "
      f"image={CFG['image_size']} crop_mode={CFG['crop_mode']} | train augment={CFG['augment']}")


In [ ]:
# ============================================================================
# Cell 7 — Datasets, measured sequence lengths, and a forward/backward smoke test
# ============================================================================
import torch
from torch.utils.data import Dataset

class OCRDataset(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self):        return len(self.rows)
    def __getitem__(self, i): return to_conversation(self.rows[i])

train_ds, val_ds = OCRDataset(TRAIN), OCRDataset(VAL)
print(f"datasets: train {len(train_ds)}  val {len(val_ds)}")

# ---- measure real sequence / target lengths instead of assuming them -------
tgt_tok = [len(text_encode(tokenizer, r["text"], bos=False, eos=False)) for r in TRAIN_ALL + TEST]
tgt_tok.sort()
q = lambda v, p: v[min(len(v) - 1, int(p * len(v)))]
MAX_TARGET_TOKENS = tgt_tok[-1]
print(f"\ntarget length in tokens (all 1135 rows): min {tgt_tok[0]}  med {q(tgt_tok,.5)}  "
      f"p95 {q(tgt_tok,.95)}  max {MAX_TARGET_TOKENS}")

# generation budget derived from the measured distribution, never from intuition
EVAL_MAX_NEW_TOKENS = CFG["eval_max_new_tokens"] or int(2 * MAX_TARGET_TOKENS + 16)
print(f"EVAL_MAX_NEW_TOKENS = {EVAL_MAX_NEW_TOKENS}  (2x the longest target + 16)")

n_probe = min(24, len(TRAIN))
seq_lens = []
for i in range(n_probe):
    b = eval_collator([train_ds[i]])
    seq_lens.append(b["input_ids"].shape[1])
seq_lens.sort()
print(f"full sequence length over {n_probe} probed samples: min {seq_lens[0]}  "
      f"med {q(seq_lens,.5)}  max {seq_lens[-1]}")
MAXPOS = getattr(model.config, "max_position_embeddings", None) or 8192
print(f"model max_position_embeddings = {MAXPOS} -> headroom OK: {seq_lens[-1] < MAXPOS}")
assert seq_lens[-1] < MAXPOS, "sequences exceed the model's position budget"

# ---- the model's forward() must accept exactly what the collator emits -------
import inspect
_base_fwd = inspect.signature(model.get_base_model().forward).parameters
_needed   = {"input_ids", "attention_mask", "labels", "images", "images_seq_mask",
             "images_spatial_crop"}
_missing  = _needed - set(_base_fwd)
print(f"\nforward() accepts every collator key: {not _missing}"
      + (f"   MISSING: {_missing}" if _missing else ""))
assert not _missing, (
    f"forward() does not accept {_missing}. The installed transformers version or the "
    f"remote modeling code has changed; Trainer would fail deep inside the training loop.")
# forward() shifts labels internally (shift_logits/shift_labels), so labels must NOT be
# pre-shifted by the collator -- and they are not.

# ---- one batch through forward+backward, before committing to a long run ----
# Deliberately use the LARGEST training image(s). VRAM peaks on the samples that tile into
# the most 640x640 crops (up to 9), and gradient checkpointing does NOT cover the custom
# vision encoder (ImageEncoderViT / VitModel do not implement HF's checkpointing hook), so
# the vision activations for those crops are the binding memory constraint. Testing the
# worst case here means an OOM surfaces in seconds, not 300 steps into training.
from PIL import Image as _PILImage
_areas = []
for _i, _r in enumerate(TRAIN):
    _w, _h = _PILImage.open(_r["image"]).size
    _areas.append((_i, _w * _h, _w, _h))
_worst = sorted(_areas, key=lambda x: -x[1])[:CFG["per_device_batch"]]
print("\n--- forward/backward smoke test (worst-case samples) ---")
for _i, _a, _w, _h in _worst:
    print(f"  {TRAIN[_i]['id']}: {_w}x{_h}px  (aspect {_w/_h:.1f})")
batch = train_collator([train_ds[i] for i, _, _, _ in _worst])
print(f"  image crops in batch: {[tuple(c.shape) for c, o in batch['images']]}")
print(f"  sequence length     : {batch['input_ids'].shape[1]}  <- worst case")
print("batch shapes:", {k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__)
                        for k, v in batch.items()})
n_sup = int((batch["labels"] != -100).sum())
print(f"supervised (unmasked) label positions: {n_sup}  "
      f"-> should be ~= target tokens + 2, and NOT include image tokens")
assert n_sup > 0, "every label is masked; the collator masked the answer too"

dev = next(p.device for p in model.parameters() if p.requires_grad)
model.train()
mb = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in batch.items()}
mb["images"] = [(c.to(dev), o.to(dev)) for c, o in batch["images"]]
try:
    with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        out = model(**mb)
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError(
        f"OOM on the largest sample (T4 has ~14.6 GB usable).\n{e}\n\n"
        f"Try in this order, and record which you used:\n"
        f"  1. CFG['adapt_vision_tower'] = False  -- drops the vision LoRA, so no gradients\n"
        f"     or optimiser state for the un-checkpointed vision encoder. Biggest saving.\n"
        f"  2. CFG['grad_accum'] = 16 with per_device_batch = 1 (already 1) -- same effective\n"
        f"     batch, lower peak.\n"
        f"  3. Only as a last resort, lower base_size/image_size -- this DEVIATES from the\n"
        f"     resolution the warm-start adapter was trained at (rule 18) and must be stated\n"
        f"     in the results."
    ) from e
loss = out.loss if hasattr(out, "loss") else out["loss"]
print(f"loss = {loss.item():.4f}")
assert torch.isfinite(loss), "loss is NaN/Inf on the very first batch"
loss.backward()
g = [(n, float(p.grad.abs().mean())) for n, p in model.named_parameters()
     if p.requires_grad and p.grad is not None and p.grad.abs().sum() > 0]
print(f"parameters that received a non-zero gradient: {len(g)}")
assert len(g) > 0, "no gradients reached the LoRA parameters"
print("  e.g.", g[:3])
model.zero_grad(set_to_none=True)
print(f"peak VRAM after smoke test: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")
del out, loss, mb; gc.collect(); torch.cuda.empty_cache()
print("smoke test PASSED")


In [ ]:
# ============================================================================
# Cell 8 — Inference with exact train/eval parity, and the metric module
# ============================================================================
import torch, re, contextlib, inspect

# ---------------------------------------------------------------- inference --
# We do NOT call model.infer(): it returns text only with eval_mode=True, hardcodes
# max_new_tokens=8192 and no_repeat_ngram_size, and applies format_messages (which the
# training collator does not). Building the prompt with the collator's own code path
# instead makes training and evaluation byte-identical.
@torch.no_grad()
def ocr_predict(row_or_path, max_new_tokens=None, use_adapter=True):
    path = row_or_path["image"] if isinstance(row_or_path, dict) else row_or_path
    msgs = [{"role": "<|User|>", "content": CFG["prompt"], "images": [path]},
            {"role": "<|Assistant|>", "content": ""}]
    s = eval_collator.process_single_sample(msgs, augment=False)

    # drop the assistant part: keep BOS + prompt + image tokens, i.e. everything the
    # collator would have masked out of the labels.
    n_prompt   = s["prompt_token_count"]
    input_ids  = s["input_ids"][:n_prompt].unsqueeze(0).to(DEVICE)
    seq_mask   = s["images_seq_mask"][:n_prompt].unsqueeze(0).to(DEVICE)
    images     = [(s["images_crop"].to(DEVICE), s["images_ori"].to(DEVICE))]

    gen_kwargs = dict(
        images              = images,
        images_seq_mask     = seq_mask,
        images_spatial_crop = s["images_spatial_crop"],
        do_sample           = False,   # greedy. NOT temperature=0.0, which generate() rejects
        num_beams           = 1,
        eos_token_id        = tokenizer.eos_token_id,
        pad_token_id        = tokenizer.pad_token_id,
        max_new_tokens      = max_new_tokens or EVAL_MAX_NEW_TOKENS,
        use_cache           = True,    # matches the reference infer() implementation
    )
    model.eval()
    ctx = model.disable_adapter() if (not use_adapter and hasattr(model, "disable_adapter")) \
          else contextlib.nullcontext()
    with ctx, torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        try:
            out = model.generate(input_ids, **gen_kwargs)
        except TypeError as e:
            # The PEFT wrapper rejected the custom multimodal kwargs. LoRA layers are
            # swapped in place inside the base module, so calling it directly still
            # applies the adapter -- it is equivalent, not a silent downgrade.
            if not getattr(ocr_predict, "_warned", False):
                print(f"[warn] wrapper generate() rejected custom kwargs ({e}); "
                      f"using the base module directly (LoRA still applies).")
                ocr_predict._warned = True
            out = model.get_base_model().generate(input_ids, **gen_kwargs)
    gen = out[0, input_ids.shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

# --------------------------------------------------------- post-processing ---
# Only removes wrappers the VLM may emit around the answer. It never edits the
# characters themselves, so it cannot flatter the model on a character metric.
_STOP = "<｜end▁of▁sentence｜>"
def postprocess(text: str) -> str:
    if text is None:
        return ""
    t = text
    for s in (_STOP, "<|end▁of▁sentence|>", "</s>"):
        t = t.replace(s, "")
    t = re.sub(r"<\|[^>]*\|>", "", t)          # grounding / ref markers, e.g. <|ref|>
    t = re.sub(r"^\s*```[a-zA-Z]*|```\s*$", "", t)
    t = t.replace("\u200b", "")               # zero-width space (NOT ZWJ U+200D, which is real Sinhala)
    t = re.sub(r"\s+", " ", t)                 # collapse newlines/multiple spaces
    return t.strip()

# ---------------------------------------------------------------- metrics ----
def edit_counts(ref: str, hyp: str):
    """Levenshtein with an operation breakdown. Returns (S, D, I, C, N).

    N = len(ref) in Unicode code points, matching IJDAR Equation 1
    (CER = (S+D+I)/N, N = S+D+C = number of reference characters).
    """
    r, h = list(ref), list(hyp)
    n, m = len(r), len(h)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): d[i][0] = i
    for j in range(m + 1): d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1,
                          d[i-1][j-1] + (r[i-1] != h[j-1]))
    S = D = I = C = 0
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and r[i-1] == h[j-1] and d[i][j] == d[i-1][j-1]:
            C += 1; i -= 1; j -= 1
        elif i > 0 and j > 0 and d[i][j] == d[i-1][j-1] + 1:
            S += 1; i -= 1; j -= 1
        elif i > 0 and d[i][j] == d[i-1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    return S, D, I, C, n

def word_edit_distance(ref: str, hyp: str):
    r, h = ref.split(), hyp.split()
    n, m = len(r), len(h)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): d[i][0] = i
    for j in range(m + 1): d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + (r[i-1] != h[j-1]))
    return d[n][m], n

def score(pairs):
    """pairs: list of (reference, hypothesis). Returns every metric variant."""
    per, tot_e, tot_n, tot_we, tot_wn, exact = [], 0, 0, 0, 0, 0
    for ref, hyp in pairs:
        S, D, I, C, N = edit_counts(ref, hyp)
        e = S + D + I
        per.append(e / N if N else (0.0 if not hyp else 1.0))
        tot_e += e; tot_n += N
        we, wn = word_edit_distance(ref, hyp)
        tot_we += we; tot_wn += wn
        exact += int(ref.strip() == hyp.strip())
    return {
        "n": len(pairs),
        "cer_corpus": tot_e / tot_n if tot_n else float("nan"),   # PRIMARY: IJDAR Eq. 1
        "cer_macro":  float(np.mean(per)) if per else float("nan"),
        "wer_corpus": tot_we / tot_wn if tot_wn else float("nan"),
        "exact_match": exact / len(pairs) if pairs else float("nan"),
        "empty_preds": sum(1 for _, h in pairs if not h.strip()),
        "per_sample_cer": per,
    }

def bootstrap_ci(pairs, n_boot=2000, seed=0, alpha=0.05):
    """95% CI on corpus CER by resampling samples (not characters)."""
    counts = []
    for ref, hyp in pairs:
        S, D, I, C, N = edit_counts(ref, hyp)
        counts.append((S + D + I, N))
    rs = np.random.RandomState(seed)
    idx = np.arange(len(counts))
    vals = []
    e = np.array([c[0] for c in counts], dtype=float)
    nn = np.array([c[1] for c in counts], dtype=float)
    for _ in range(n_boot):
        s = rs.choice(idx, size=len(idx), replace=True)
        den = nn[s].sum()
        vals.append(e[s].sum() / den if den else np.nan)
    lo, hi = np.nanpercentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

def evaluate(rows, label, max_new_tokens=None, use_adapter=True, show=3, save_as=None):
    t0 = time.time()
    recs = []
    for k, r in enumerate(rows):
        raw = ocr_predict(r, max_new_tokens=max_new_tokens, use_adapter=use_adapter)
        hyp = postprocess(raw)
        recs.append({"id": r["id"], "reference": r["text"], "prediction": hyp, "raw": raw,
                     "seen_in_train_all": r.get("seen_in_train_all")})
        if k < show:      # rule: always print raw samples, not just aggregates
            print(f"  [{k}] ref : {r['text']}")
            print(f"      pred: {hyp}")
    pairs = [(x["reference"], x["prediction"]) for x in recs]
    m = score(pairs)
    lo, hi = bootstrap_ci(pairs, seed=CFG["seed"])
    m["cer_ci95"] = [lo, hi]
    m["seconds"] = time.time() - t0
    print(f"\n{label}: CER(corpus) {m['cer_corpus']:.4f}  [95% CI {lo:.4f}-{hi:.4f}]   "
          f"CER(macro) {m['cer_macro']:.4f}   WER {m['wer_corpus']:.4f}   "
          f"exact {m['exact_match']:.4f}   n={m['n']}   {m['seconds']/60:.1f} min")
    if m["empty_preds"]:
        print(f"  [warn] {m['empty_preds']}/{m['n']} predictions were EMPTY. A CER pinned near 1.0 "
              f"with no variance means a broken pipeline, not a weak model.")
    if save_as:
        with open(os.path.join(RUN_DIR, save_as), "w", encoding="utf-8") as f:
            json.dump({"label": label, "metrics": {k: v for k, v in m.items()
                                                   if k != "per_sample_cer"},
                       "records": recs}, f, ensure_ascii=False, indent=2)
    return m, recs

DEVICE = next(p.device for p in model.parameters() if p.requires_grad)

# ---- prove train/eval prompt parity (the trailing-space trap) --------------
_probe = TRAIN[0]
_train_ids = train_collator.process_single_sample(to_conversation(_probe)["messages"],
                                                  augment=False)
_eval_msgs = [{"role": "<|User|>", "content": CFG["prompt"], "images": [_probe["image"]]},
              {"role": "<|Assistant|>", "content": ""}]
_eval_ids = eval_collator.process_single_sample(_eval_msgs, augment=False)
_a = _train_ids["input_ids"][:_train_ids["prompt_token_count"]].tolist()
_b = _eval_ids["input_ids"][:_eval_ids["prompt_token_count"]].tolist()
print(f"train prompt tokens == eval prompt tokens : {_a == _b}  (len {len(_a)} vs {len(_b)})")
assert _a == _b, "train/eval prompt mismatch — the model would be evaluated on an unseen prefix"

# ---- sanity-check the metric against jiwer, and against the paper's formula ----
try:
    import jiwer
    _ref, _hyp = "හෘද සෑත්කමක්", "හෘද සෑත්"
    S, D, I, C, N = edit_counts(_ref, _hyp)
    print(f"metric check: ours (S+D+I)/N = {(S+D+I)/N:.6f}   jiwer.cer = {jiwer.cer(_ref, _hyp):.6f}")
except Exception as e:
    print(f"(jiwer cross-check skipped: {e})")


In [ ]:
# ============================================================================
# Cell 9 — Zero-shot baselines, measured BEFORE any training
# ============================================================================
# Two rows, both on the same test samples with the same metric:
#   (a) base DeepSeek-OCR V1 with LoRA disabled  -> the un-adapted model
#   (b) the Sinhala PRINT adapter as loaded       -> the paper's printed-text model, on handwriting
# (b) is the direct analogue of IJDAR's "TrOCR trained on printed only" row (0.9940), so it
# measures how much print-only Sinhala training transfers to handwriting.
ZS = {}
if CFG["run_zero_shot"]:
    zs_rows = TEST[:CFG["zeroshot_n"]]
    print(f"Zero-shot baselines on {len(zs_rows)} test samples "
          f"(max_new_tokens={CFG['zeroshot_max_new_tok']} to bound rambling)\n")

    print("--- (a) base DeepSeek-OCR V1, LoRA disabled ---")
    ZS["base_v1"] = evaluate(zs_rows, "zero-shot base V1",
                             max_new_tokens=CFG["zeroshot_max_new_tok"],
                             use_adapter=False, save_as="zeroshot_base_v1.json")[0]

    if WARM_START_OK:
        print("\n--- (b) Sinhala PRINT adapter (arXiv:2606.29378 Exp.1), un-finetuned on handwriting ---")
        ZS["print_adapter"] = evaluate(zs_rows, "zero-shot Sinhala print adapter",
                                       max_new_tokens=CFG["zeroshot_max_new_tok"],
                                       use_adapter=True, save_as="zeroshot_print_adapter.json")[0]
    with open(os.path.join(RUN_DIR, "zeroshot_summary.json"), "w") as f:
        json.dump({k: {kk: vv for kk, vv in v.items() if kk != "per_sample_cer"}
                   for k, v in ZS.items()}, f, indent=2)
    gc.collect(); torch.cuda.empty_cache()
else:
    print("run_zero_shot=False -> skipping baselines")


In [ ]:
# ============================================================================
# Cell 10 — Train (with validation-CER early stopping and a wall-clock guard)
# ============================================================================
from transformers import Trainer, TrainingArguments, TrainerCallback

class ValCERCallback(TrainerCallback):
    """Scores validation CER by real generation, keeps the best adapter, stops early.

    Validation uses generation (not teacher-forced loss) because loss and CER can move
    independently: falling loss with flat CER means the decoder is fitting tokens but not
    producing usable text. Selecting on CER selects on the metric we report.
    """
    def __init__(self, rows, every, n, patience):
        self.rows, self.every, self.n, self.patience = rows, every, n, patience
        self.best, self.bad, self.history = float("inf"), 0, []

    def _score(self, state):
        rows = self.rows[:self.n]
        pairs = []
        for k, r in enumerate(rows):
            hyp = postprocess(ocr_predict(r))
            pairs.append((r["text"], hyp))
            if k < 2:
                print(f"    val[{k}] ref : {r['text']}")
                print(f"            pred: {hyp}")
        m = score(pairs)
        self.history.append({"step": state.global_step, "cer": m["cer_corpus"],
                             "exact": m["exact_match"], "empty": m["empty_preds"]})
        print(f"  >> step {state.global_step}: val CER {m['cer_corpus']:.4f}  "
              f"exact {m['exact_match']:.3f}  empty {m['empty_preds']}/{m['n']}")
        return m["cer_corpus"]

    def on_step_end(self, args, state, control, **kw):
        if state.global_step > 0 and state.global_step % self.every == 0:
            cer = self._score(state)
            if cer < self.best - 1e-4:
                self.best, self.bad = cer, 0
                kw["model"].save_pretrained(BEST_DIR)
                print(f"     new best -> saved to {BEST_DIR}")
            else:
                self.bad += 1
                print(f"     no improvement ({self.bad}/{self.patience}); best {self.best:.4f}")
                if self.bad >= self.patience:
                    print("     early stopping")
                    control.should_training_stop = True
            model.train()
        return control

class WallClockGuard(TrainerCallback):
    """Kaggle sessions are capped; stop training in time for the evaluation cells."""
    def __init__(self, hours): self.deadline = time.time() + hours * 3600; self.hours = hours
    def on_step_end(self, args, state, control, **kw):
        if time.time() > self.deadline:
            print(f"\n[budget] {self.hours} h training budget reached at step {state.global_step}; stopping.")
            control.should_training_stop = True
        return control

if CFG["do_train"]:
    steps_per_epoch = max(1, len(TRAIN) // (CFG["per_device_batch"] * CFG["grad_accum"]))
    print(f"~{steps_per_epoch} optimizer steps/epoch x {CFG['epochs']} epochs "
          f"= ~{steps_per_epoch * CFG['epochs']} steps (effective batch "
          f"{CFG['per_device_batch'] * CFG['grad_accum']})")

    val_cb = ValCERCallback(VAL, CFG["eval_every_steps"], CFG["val_eval_n"], CFG["early_stop_patience"])
    args = TrainingArguments(
        output_dir                  = os.path.join(RUN_DIR, "trainer"),
        num_train_epochs            = CFG["epochs"],
        per_device_train_batch_size = CFG["per_device_batch"],
        gradient_accumulation_steps = CFG["grad_accum"],
        learning_rate               = CFG["lr"],
        warmup_ratio                = CFG["warmup_ratio"],
        weight_decay                = CFG["weight_decay"],
        max_grad_norm               = CFG["max_grad_norm"],
        lr_scheduler_type           = "cosine",
        logging_steps               = 5,
        eval_strategy               = "no",     # we run our own generation-based validation
        save_strategy               = "no",     # the callback saves the best adapter itself
        fp16                        = USE_FP16_AMP,
        bf16                        = not USE_FP16_AMP,
        optim                       = "paged_adamw_8bit",
        remove_unused_columns       = False,    # REQUIRED: our features are not model kwargs
        dataloader_num_workers      = 2,
        report_to                   = "none",
        seed                        = CFG["seed"],
        gradient_checkpointing      = False,    # already enabled explicitly in Cell 5
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      data_collator=train_collator,
                      callbacks=[val_cb, WallClockGuard(CFG["train_hours_budget"])])

    t0 = time.time()
    stats = trainer.train()
    print(f"\ntrained in {(time.time()-t0)/60:.1f} min | {stats.metrics}")
    print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

    with open(os.path.join(RUN_DIR, "train_history.json"), "w") as f:
        json.dump({"val_history": val_cb.history, "best_val_cer": val_cb.best,
                   "log_history": trainer.state.log_history, "config": {k: str(v) for k, v in CFG.items()}},
                  f, indent=2)

    # Load the best checkpoint by validation CER (never by test CER).
    if os.path.isdir(BEST_DIR) and val_cb.best < float("inf"):
        from peft import PeftModel
        print(f"\nreloading best adapter (val CER {val_cb.best:.4f}) from {BEST_DIR}")
        sd = load_file(os.path.join(BEST_DIR, "adapter_model.safetensors"))
        r = set_peft_model_state_dict(model, sd, adapter_name="default")
        bad = [k for k in r.unexpected_keys if "lora_" in k]
        assert not bad, f"could not restore best adapter: {bad[:3]}"
        print("best adapter restored")
    else:
        print("\n[warn] no best checkpoint was saved; evaluating the final weights instead.")
    gc.collect(); torch.cuda.empty_cache()
else:
    print("do_train=False -> skipping training")


In [ ]:
# ============================================================================
# Cell 11 — Final evaluation on the held-out test split (touched only here)
# ============================================================================
if CFG["run_final_test_eval"]:
    print(f"Evaluating on all {len(TEST)} test samples "
          f"(first and only use of the test split)\n")
    m_ft, recs = evaluate(TEST, "FINE-TUNED DeepSeek-OCR + QLoRA", show=6,
                          save_as="final_test_predictions.json")

    # ---- seen-text vs unseen-text breakdown ------------------------------
    seen   = [(r["reference"], r["prediction"]) for r in recs if r["seen_in_train_all"]]
    unseen = [(r["reference"], r["prediction"]) for r in recs if not r["seen_in_train_all"]]
    m_seen   = score(seen)   if seen   else None
    m_unseen = score(unseen) if unseen else None

    # ---- per-length buckets ---------------------------------------------
    buckets = {"1-9 chars": [], "10-19": [], "20-34": [], "35+": []}
    for r in recs:
        n = len(r["reference"])
        k = "1-9 chars" if n < 10 else "10-19" if n < 20 else "20-34" if n < 35 else "35+"
        buckets[k].append((r["reference"], r["prediction"]))

    IJDAR = {"TrOCR (printed only)": 0.9940, "Tesseract (pre-trained)": 0.9493,
             "Google Vision API": 0.7532, "Tesseract (printed+handwritten)": 0.7204,
             "TrOCR (printed->handwritten)": 0.5253}

    print("\n" + "=" * 78)
    print("RESULTS — SinOCR-Handwritten test split (n=227), CER = (S+D+I)/N on code points")
    print("=" * 78)
    print(f"{'Model':44} {'CER':>8} {'source':>22}")
    print("-" * 78)
    for k, v in sorted(IJDAR.items(), key=lambda x: -x[1]):
        print(f"{k:44} {v:8.4f} {'IJDAR Table 5':>22}")
    for k, label in (("base_v1", "DeepSeek-OCR V1 zero-shot"),
                     ("print_adapter", "DeepSeek-OCR V1 + Sinhala PRINT LoRA (zero-shot)")):
        if k in ZS:
            print(f"{label:44} {ZS[k]['cer_corpus']:8.4f} {'this notebook':>22}")
    print(f"{'DeepSeek-OCR V1 + QLoRA (handwritten) [OURS]':44} {m_ft['cer_corpus']:8.4f} "
          f"{'this notebook':>22}")
    print("-" * 78)
    delta = IJDAR["TrOCR (printed->handwritten)"] - m_ft["cer_corpus"]
    rel   = 100 * delta / IJDAR["TrOCR (printed->handwritten)"]
    verdict = "BEATS" if delta > 0 else "does NOT beat"
    print(f"vs the TrOCR bar (0.5253): {verdict} it by {abs(delta):.4f} CER "
          f"({abs(rel):.1f}% relative)")
    print(f"95% CI on our CER: [{m_ft['cer_ci95'][0]:.4f}, {m_ft['cer_ci95'][1]:.4f}]"
          f"  -> improvement is {'significant' if m_ft['cer_ci95'][1] < 0.5253 else 'NOT clearly significant'}"
          f" at the 95% level")

    print("\nSecondary metrics and breakdowns")
    print("-" * 78)
    print(f"  CER macro-average (mean of per-sample CER) : {m_ft['cer_macro']:.4f}")
    print(f"  WER (corpus)                               : {m_ft['wer_corpus']:.4f}")
    print(f"  exact-match rate                           : {m_ft['exact_match']:.4f}")
    print(f"  empty predictions                          : {m_ft['empty_preds']}/{m_ft['n']}")
    if m_seen and m_unseen:
        print(f"  CER on seen-text rows   (n={m_seen['n']:3})            : {m_seen['cer_corpus']:.4f}")
        print(f"  CER on unseen-text rows (n={m_unseen['n']:3})            : {m_unseen['cer_corpus']:.4f}")
        print("    (both models in the comparison face the same published split; the unseen-text")
        print("     column is the honest estimate for genuinely new vocabulary)")
    print("  by reference length:")
    for k, v in buckets.items():
        if v:
            print(f"    {k:12} n={len(v):3}  CER {score(v)['cer_corpus']:.4f}")

    # ---- worst cases, for qualitative error analysis ---------------------
    def _sample_cer(r):
        S, D, I, C, N = edit_counts(r["reference"], r["prediction"])
        return (S + D + I) / max(1, N)
    worst = sorted(recs, key=_sample_cer, reverse=True)[:8]
    print("\nWorst 8 predictions (for error analysis):")
    for r in worst:
        S, D, I, C, N = edit_counts(r["reference"], r["prediction"])
        print(f"  CER {(S+D+I)/max(1,N):.2f}  ref: {r['reference']}")
        print(f"            pred: {r['prediction']}")

    # ---- persist everything ---------------------------------------------
    report = {
        "dataset": {"name": "SinOCR-Handwritten (Datasets/handwritten-data)",
                    "train_used": len(TRAIN), "val": len(VAL), "test": len(TEST),
                    "identity_verified_vs_IJDAR_table2": True},
        "metric": "CER = (S+D+I)/N, N = reference length in Unicode code points (IJDAR Eq. 1)",
        "config": {k: str(v) for k, v in CFG.items()},
        "results": {
            "ours_finetuned": {k: v for k, v in m_ft.items() if k != "per_sample_cer"},
            "zero_shot": {k: {kk: vv for kk, vv in v.items() if kk != "per_sample_cer"}
                          for k, v in ZS.items()},
            "ijdar_table5_baselines": IJDAR,
            "seen_text":   ({k: v for k, v in m_seen.items()   if k != "per_sample_cer"} if m_seen else None),
            "unseen_text": ({k: v for k, v in m_unseen.items() if k != "per_sample_cer"} if m_unseen else None),
            "by_length": {k: score(v)["cer_corpus"] for k, v in buckets.items() if v},
        },
    }
    with open(os.path.join(RUN_DIR, "RESULTS.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    model.save_pretrained(os.path.join(RUN_DIR, "final_adapter"))
    print(f"\nsaved: {RUN_DIR}/RESULTS.json, final_test_predictions.json, final_adapter/, best_adapter/")
    print("Download the whole deepseek_hw_run/ folder from Kaggle's output before the session ends.")
else:
    print("run_final_test_eval=False -> skipping")


## How to read the output, and what to do next

### Reading the results
- **`CER(corpus)`** is the headline number: `(S+D+I)/N` over the whole test set, `N` in Unicode
  code points — the same definition as IJDAR Equation 1, so it is directly comparable to **0.5253**.
- `CER(macro)` is also printed because the IJDAR paper does not state whether it pooled errors or
  averaged per sample. Reporting both means the comparison holds under either reading.
- The **95% CI** decides whether an improvement is real. With n=227, a CER of ~0.45 with a CI
  reaching past 0.5253 is *not* yet a defensible win.
- Check **`empty predictions`** first if CER looks suspiciously close to 1.0 with little variance —
  that indicates a broken pipeline, not a weak model.

### If the first run underperforms, change these in Cell 1, in this order
1. **`epochs`** — if val CER was still falling when the wall-clock guard or early stopping fired,
   more epochs is the cheapest win. Check `train_history.json`.
2. **`adapt_vision_tower = False`** — isolates whether vision adaptation helped or destabilised
   training. This is the single biggest architectural unknown here.
3. **`warm_start = False`** — the ablation that tests the two-stage premise itself. If a fresh LoRA
   beats the warm-started one, print pre-training did *not* transfer, which is a finding worth
   reporting either way.
4. **`lr`** — try `5e-5` if the loss curve is jagged, `2e-4` if it barely moved.
5. **`augment = False`** — checks whether augmentation is helping or blurring away diacritics.
6. **`lora_dropout`** — raise toward `0.1` if val CER worsens while train loss keeps falling
   (classic overfitting on ~800 samples).

Each of these is a legitimate ablation row for the thesis, not just a knob — record the CER for each.

### What to bring back for the next session
- `deepseek_hw_run/RESULTS.json` (all metrics + config)
- `deepseek_hw_run/train_history.json` (loss curve + validation CER trajectory)
- `deepseek_hw_run/final_test_predictions.json` (per-sample predictions for error analysis)
- the console log, especially Cell 5's warm-start verification, Cell 7's smoke test, and the
  first validation block

### Honest caveats to state in the write-up
- This run's numbers are **reasoned, not yet verified** — the notebook has never been executed,
  because this project has no local GPU. Treat the first Kaggle run as the real test.
- ~34% of test rows share their exact text with a train row. That is a property of the *published*
  split and TrOCR's 0.5253 was measured under identical conditions, so the comparison is fair — but
  the unseen-text column is the better estimate of performance on new vocabulary, and both are reported.
- Validation for checkpoint selection is carved out of train; the test split is read exactly once,
  in Cell 11. No number in the results table was selected on test.
